# CS17 — 4 group models → clean 3D label volume → light web bundle for `<embryo-viewer>`

**Input:** the registered grey volume from the registration notebook (`volumes/<SPECIMEN>/` in the shared Hugging Face dataset) + the 4 nnU-Net group checkpoints.
**No Atlas label volume and no Atlas names are used.** Structures are identified by their numeric Atlas ID only (IDs are never renumbered); names come later from `names.json`.

**Output (no HTML, no GLB):** `viewers/<SPECIMEN>/` in the same shared dataset
- `meshes.bin` — all structures, quantised + delta-coded + zlib, coarse level of detail first so the page shows the whole embryo almost immediately
- `manifest.json` — structures, colours, volumes, byte offsets
- `names.json` — `{"<atlas id>": "Name"}`; **never overwritten** if it already exists in the dataset (edit it any time, no rebuild needed)

The page side is the `embryo-viewer.js` web component (separate file, hosted on your website).

Run on a **Kaggle GPU** session with Internet ON and `HF_TOKEN` in Secrets. Set `SMOKE_TEST_SLABS` in the config to a small range first to check the whole chain in a few minutes.


In [1]:
# ============================================================
# 0. CONFIGURATION
# ============================================================
import os, sys, json, re, time, shutil, subprocess, gc, zlib, math, hashlib
from pathlib import Path
import numpy as np

STAGE = "CS17"
ATLAS_SPECIMEN_ID = "6521"
SPECIMEN_ID = f"{STAGE}-{ATLAS_SPECIMEN_ID}"
TAG = SPECIMEN_ID.replace("-", "_")

# --- Hugging Face ---
HF_SHARED_REPO_ID = "senku111/embryo-3d"                              # volumes/<id>/ (input)  +  viewers/<id>/ (output). Must be PUBLIC for the website.
HF_DATASET_REPO_ID = "senku111/atlas-cs17-4groups-dataset"            # holds all_groups_manifest.json (atlas ids per group, normalisation)
HF_TRAINING_REPO_ID = "senku111/atlas-cs17-4groups-nnunet-training"   # holds the 4 checkpoints (model repo)
N_GROUPS = 4

# --- checkpoint choice per group ---
CHECKPOINT_POLICY = "auto"       # "auto": final if it exists else best | "final" | "best" | "latest"
CHECKPOINT_OVERRIDE = {}         # e.g. {2: "latest"}
DISABLE_TTA = True               # True = ~4-8x faster inference. Set False for the final high-quality run if time allows.

# --- resolution of the 3D label volume ---
MAX_VOXELS = 600_000_000         # uint8 labels -> 600 MB RAM. (the old notebook: 150M as uint16)
INFER_ALL_SLICES = True          # predict every slice and pool z-slabs by coverage (only matters when z is down-sampled)
FG_THRESHOLD = 0.40              # a label wins a down-sampled voxel when it covers >= 40% of it (keeps thin structures alive)
SMOKE_TEST_SLABS = None          # e.g. (400, 460) -> only reconstruct slabs 400..459 (fast end-to-end check). None = everything.

# --- meshing ---
MIN_VOXELS = 40                  # ignore structures smaller than this (working-resolution voxels)
SMOOTH_MM = 0.014                # physical gaussian sigma applied to each structure before marching cubes
Z_EXTRA = 1.5                    # extra smoothing along z (the models predict slices independently)
TAUBIN_ITERS = 12
MESH_MAX_CROP = 30_000_000       # structures with a bigger bounding box are meshed on a coarser grid
MAX_FACES = 40_000               # hard cap per structure (fine level of detail)
MIN_FACES = 1_200
FACES_PER_VOX23 = 5.0            # target faces ~ this * voxels^(2/3), clamped to [MIN_FACES, MAX_FACES]
LOD0_FRACTION = 0.10             # coarse level (shown first) = 10% of the fine faces ...
LOD0_MIN_FACES = 150             # ... but at least this many; structures smaller than 3x this get a single level
AO_RADIUS_MM = 0.05              # ambient-occlusion blur radius (baked per vertex)
AO_OFFSET_MM = 0.02
AO_BIAS, AO_GAIN = 0.28, 1.8
AO_STRENGTH = 0.6                # how strongly the viewer applies the baked AO

PUSH_TO_HF = True

BASE = Path(f"/kaggle/working/cs17_recon_{TAG}")
RAW, INFER, PRED, MODELS, OUT = (BASE / d for d in ["raw", "inference/imagesTs", "inference/pred", "models", "bundle"])
for d in [RAW, INFER, PRED, MODELS, OUT]:
    d.mkdir(parents=True, exist_ok=True)
print("work dir:", BASE)


work dir: /kaggle/working/cs17_recon_CS17_6521


In [2]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nnunetv2", "huggingface_hub", "tifffile", "imagecodecs",
                "trimesh", "fast_simplification", "scikit-image", "scipy", "pillow", "opencv-python-headless"], check=True)
import torch
assert torch.cuda.is_available(), "No GPU. Turn on a Kaggle GPU for the inference stage."

def get_hf_token():
    t = os.environ.get("HF_TOKEN")
    if t: return t
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None
HF_TOKEN = get_hf_token()
assert HF_TOKEN, "HF_TOKEN missing (Kaggle Secrets). Needed for the private checkpoint repo and for pushing the bundle."
print("GPU:", torch.cuda.get_device_name(0))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.4/750.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 286.8/286.8 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 4.1 MB/s eta 0:00:00
GPU: Tesla T4


## 1. Get the registered volume + its manifest from the shared dataset
The voxel size and shape come from the registration manifest (no Atlas scraping), and the file is verified against its SHA-256.
The intensity normalisation is the **same global range the training data used** (from `all_groups_manifest.json`), not a per-slice stretch.


In [3]:
import tifffile, cv2
from PIL import Image
from huggingface_hub import hf_hub_download, HfApi
api = HfApi(token=HF_TOKEN)

def _dl(repo, fn, rtype="dataset", local=RAW):
    return hf_hub_download(repo, fn, repo_type=rtype, token=HF_TOKEN, local_dir=str(local))

vol_manifest_path = _dl(HF_SHARED_REPO_ID, f"volumes/{SPECIMEN_ID}/{SPECIMEN_ID}_manifest.json")
VMAN = json.load(open(vol_manifest_path))
GREY_TIF = Path(_dl(HF_SHARED_REPO_ID, f"volumes/{SPECIMEN_ID}/{SPECIMEN_ID}_grey.tif"))

if VMAN.get("volume_sha256"):
    h = hashlib.sha256()
    with open(GREY_TIF, "rb") as f:
        for blk in iter(lambda: f.read(1 << 24), b""): h.update(blk)
    assert h.hexdigest() == VMAN["volume_sha256"], "[DATA PROBLEM] downloaded volume does not match its manifest sha256"
    print("volume sha256 verified")
else:
    print("!! manifest has no volume_sha256 (older registration run) - integrity not verified")

Z, H, W = [int(v) for v in VMAN["shape_zyx"]]
xy_um, z_um = float(VMAN["xy_pixel_size_um"]), float(VMAN["z_spacing_um"])
print(f"volume {Z} x {H} x {W}  | voxel xy {xy_um:.3f} um, z {z_um:.3f} um | "
      f"extent {W*xy_um/1000:.2f} x {H*xy_um/1000:.2f} x {Z*z_um/1000:.2f} mm")

# groups + training normalisation
gm_path = _dl(HF_DATASET_REPO_ID, f"training_dataset/{SPECIMEN_ID}/reports/all_groups_manifest.json", local=BASE / "manifest_dl")
GM = json.load(open(gm_path))
groups = {g["group_index"]: g for g in GM["groups"]}
assert len(groups) == N_GROUPS, f"expected {N_GROUPS} groups, manifest has {len(groups)}"
NORM_LO, NORM_HI = float(GM["normalization"]["global_lo"]), float(GM["normalization"]["global_hi"])
ALL_ATLAS_IDS = sorted(int(a) for g in groups.values() for a in g["atlas_ids"])
print("groups:", {gi: g["n_classes"] for gi, g in groups.items()}, "| training normalisation:", NORM_LO, "-", NORM_HI)

# --- down-sampling plan: biggest resolution that fits MAX_VOXELS ---
def plan(dxy):
    dz = max(1, round(dxy * xy_um / z_um))
    return dz, (-(-Z // dz)) * (-(-H // dxy)) * (-(-W // dxy))
DXY = next((d for d in range(1, 17) if plan(d)[1] <= MAX_VOXELS), 16)
DZ = plan(DXY)[0]
NSLAB = -(-Z // DZ)
HS, WS = -(-H // DXY), -(-W // DXY)
SLAB_RANGE = SMOKE_TEST_SLABS or (0, NSLAB)
SLABS = [list(range(k * DZ, min(Z, (k + 1) * DZ))) if INFER_ALL_SLICES else [min(Z - 1, k * DZ)]
         for k in range(*SLAB_RANGE)]
INFER_Z = sorted({z for s in SLABS for z in s})
SPACING_MM_ZYX = (DZ * z_um / 1000, DXY * xy_um / 1000, DXY * xy_um / 1000)
print(f"label volume: xy /{DXY}, z /{DZ} -> {len(SLABS)} slabs x {HS} x {WS} = {len(SLABS)*HS*WS/1e6:.0f} M voxels "
      f"| voxel {tuple(round(v*1000,2) for v in SPACING_MM_ZYX)} um | {len(INFER_Z)} slices to predict")


CS17-6521_manifest.json:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

volumes/CS17-6521/CS17-6521_grey.tif: reconstructing file:   0%|          |  0.00B / 1.11GB            

volumes/CS17-6521/CS17-6521_grey.tif: downloading bytes:           |  0.00B            

volume sha256 verified
volume 1059 x 1821 x 1126  | voxel xy 4.479 um, z 10.000 um | extent 5.04 x 8.16 x 10.59 mm


all_groups_manifest.json:   0%|          | 0.00/3.19k [00:00<?, ?B/s]

groups: {0: 21, 1: 22, 2: 22, 3: 22} | training normalisation: 101.0 - 255.0
label volume: xy /2, z /1 -> 1059 slabs x 911 x 563 = 543 M voxels | voxel (10.0, 8.96, 8.96) um | 1059 slices to predict


## 2. Export the slices to PNG (identical normalisation to the training PNGs)


In [4]:
grey = tifffile.imread(GREY_TIF)
assert grey.shape == (Z, H, W), f"volume shape {grey.shape} != manifest {(Z, H, W)}"
denom = NORM_HI - NORM_LO
done = 0
for z in INFER_Z:
    out = INFER / f"{TAG}_{z:05d}_0000.png"
    if out.exists(): done += 1; continue
    sl = np.clip((grey[z].astype(np.float32) - NORM_LO) / denom, 0, 1)      # same formula as the data-prep notebook
    Image.fromarray((sl * 255).astype(np.uint8)).save(out)
del grey; gc.collect()
print(f"{len(INFER_Z)} slice PNGs ready in {INFER} ({done} already existed)")


1059 slice PNGs ready in /kaggle/working/cs17_recon_CS17_6521/inference/imagesTs (0 already existed)


## 3. Restore each group's checkpoint and run inference
Per group: lists what exists in the HF repo, reads epoch counts from the training logs, picks final → else best (or per `CHECKPOINT_POLICY`), detects trainer + plans name, then calls `nnUNetv2_predict`.


In [5]:
repo_files = api.list_repo_files(HF_TRAINING_REPO_ID, repo_type="model")
print(len(repo_files), "files in", HF_TRAINING_REPO_ID)

def survey_group(gi):
    g = groups[gi]
    pre = f"group{gi:02d}_{g['dataset_name']}/nnUNet_results/Dataset{g['dataset_id']:03d}_{g['dataset_name']}/"
    runs = {}
    for f in repo_files:
        if not f.startswith(pre): continue
        m = re.match(r"^([^/]+)__([^/]+)__2d/(.*)$", f[len(pre):])
        if m: runs.setdefault((m.group(1), m.group(2)), []).append(m.group(3))
    info = []
    for (tr, pl), files in runs.items():
        fold0 = {x.split("/", 1)[1] for x in files if x.startswith("fold_0/")}
        ep = -1
        for lg in sorted(x for x in files if x.startswith("fold_0/training_log")):
            p = hf_hub_download(HF_TRAINING_REPO_ID, pre + f"{tr}__{pl}__2d/" + lg, repo_type="model", token=HF_TOKEN, local_dir=str(BASE / "logs"))
            for mm in re.finditer(r"Epoch (\d+)", open(p, errors="ignore").read()): ep = max(ep, int(mm.group(1)) + 1)
        info.append(dict(trainer=tr, plans=pl, final="checkpoint_final.pth" in fold0, best="checkpoint_best.pth" in fold0,
                         latest="checkpoint_latest.pth" in fold0, epochs_logged=ep, prefix=pre))
    return info

survey = {gi: survey_group(gi) for gi in sorted(groups)}
for gi, runs in survey.items():
    print(f"GROUP {gi} ({groups[gi]['dataset_name']}, {groups[gi]['n_classes']} classes)")
    if not runs: print("  !! NOTHING FOUND for this group")
    for r in runs: print(f"  {r['trainer']}__{r['plans']}: final={r['final']} best={r['best']} latest={r['latest']} epochs_in_log={r['epochs_logged']}")

def choose(gi):
    runs = [r for r in survey[gi] if r["final"] or r["best"] or r["latest"]]
    assert runs, f"group {gi}: no usable checkpoint found"
    run = max(runs, key=lambda r: (r["final"], r["epochs_logged"]))
    pol = CHECKPOINT_OVERRIDE.get(gi, CHECKPOINT_POLICY)
    order = {"auto": ["final", "best", "latest"], "final": ["final", "best", "latest"],
             "best": ["best", "final", "latest"], "latest": ["latest", "final", "best"]}[pol]
    return run, next(k for k in order if run[k])

MODEL_INFO = {}
for gi in sorted(groups):
    g = groups[gi]; run, ck = choose(gi)
    name = f"Dataset{g['dataset_id']:03d}_{g['dataset_name']}"
    mdir = MODELS / f"group{gi:02d}" / name / f"{run['trainer']}__{run['plans']}__2d"
    for rel in ["dataset.json", "plans.json", f"fold_0/checkpoint_{ck}.pth"]:
        src = run["prefix"] + f"{run['trainer']}__{run['plans']}__2d/" + rel
        if src not in repo_files:
            src = run["prefix"] + rel
            assert src in repo_files, f"group {gi}: missing {rel} in repo"
        p = hf_hub_download(HF_TRAINING_REPO_ID, src, repo_type="model", token=HF_TOKEN, local_dir=str(BASE / "dl"))
        dst = mdir / rel; dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(p, dst)
    ep_ck = int(torch.load(mdir / f"fold_0/checkpoint_{ck}.pth", map_location="cpu", weights_only=False)["current_epoch"])
    MODEL_INFO[gi] = dict(name=name, trainer=run["trainer"], plans=run["plans"], ckpt=ck, ckpt_epoch=ep_ck,
                          results_root=str(MODELS / f"group{gi:02d}"), dataset_json=str(mdir / "dataset.json"))
    print(f"group {gi}: checkpoint_{ck}.pth (epoch {ep_ck}) | {run['trainer']} / {run['plans']}")
json.dump(MODEL_INFO, open(BASE / "model_info.json", "w"), indent=2)


136 files in senku111/atlas-cs17-4groups-nnunet-training


training_log_2026_9_30_15_20_18.txt:   0%|          | 0.00/33.7k [00:00<?, ?B/s]

training_log_2026_9_30_18_38_52.txt:   0%|          | 0.00/80.0k [00:00<?, ?B/s]

training_log_2026_9_30_23_45_07.txt:   0%|          | 0.00/162k [00:00<?, ?B/s]

training_log_2026_10_1_10_42_28.txt:   0%|          | 0.00/30.1k [00:00<?, ?B/s]

...

dataset.json:   0%|          | 0.00/759 [00:00<?, ?B/s]

plans.json:   0%|          | 0.00/6.38k [00:00<?, ?B/s]

group03_AtlasCS17FourGroup03/nnUNet_resu(…): reconstructing file:   0%|          |  0.00B /  470MB            

group03_AtlasCS17FourGroup03/nnUNet_resu(…): downloading bytes:           |  0.00B            

group 3: checkpoint_final.pth (epoch 250) | nnUNetTrainer_250epochs / nnUNetPlans_smallpatch


In [6]:
def run_group(gi):
    mi = MODEL_INFO[gi]
    out = PRED / f"group{gi:02d}"; out.mkdir(parents=True, exist_ok=True)
    have = {p.name for p in out.glob("*.png")}
    if all(f"{TAG}_{z:05d}.png" in have for z in INFER_Z):
        print(f"group {gi}: predictions already complete"); return
    env = dict(os.environ, nnUNet_results=mi["results_root"],
               nnUNet_raw=str(BASE / "nn_raw_dummy"), nnUNet_preprocessed=str(BASE / "nn_pre_dummy"))
    os.makedirs(env["nnUNet_raw"], exist_ok=True); os.makedirs(env["nnUNet_preprocessed"], exist_ok=True)
    cmd = ["nnUNetv2_predict", "-i", str(INFER), "-o", str(out), "-d", mi["name"], "-c", "2d",
           "-tr", mi["trainer"], "-p", mi["plans"], "-f", "0", "-chk", f"checkpoint_{mi['ckpt']}.pth",
           "-device", "cuda", "-npp", "2", "-nps", "2"]
    if DISABLE_TTA: cmd.append("--disable_tta")
    t = time.time(); subprocess.run(cmd, check=True, env=env)
    print(f"group {gi}: done in {(time.time()-t)/60:.1f} min")

for gi in sorted(groups):
    run_group(gi)



#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 1059 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 1059 cases that I would like to predict

Predicting CS17_6521_00000:
perform_everything_on_device: True


100%|██████████| 4/4 [00:02<00:00,  1.35it/s]


sending off prediction to background worker for resampling and export
done with CS17_6521_00000

Predicting CS17_6521_00001:
perform_everything_on_device: True


100%|██████████| 4/4 [00:00<00:00, 31.90it/s]


sending off prediction to background worker for resampling and export
done with CS17_6521_00001

Predicting CS17_6521_00002:
perform_everything_on_device: True


...

 25%|██▌       | 7/28 [00:00<00:00, 63.15it/s]

sending off prediction to background worker for resampling and export
done with CS17_6521_01057

Predicting CS17_6521_01058:
perform_everything_on_device: True


100%|██████████| 28/28 [00:00<00:00, 49.45it/s]


sending off prediction to background worker for resampling and export
done with CS17_6521_01058
GPU prediction completed. Waiting for remaining segmentation exports to finish...


Segmentation export complete.
group 3: done in 20.0 min


## 4. Merge the 4 predictions into one clean label volume (global Atlas IDs)
* Local ids → Atlas ids through each group's `atlas_ids` (the table is printed — check it once).
* Down-sampling is by **coverage** (area-averaged per label, label wins at ≥ `FG_THRESHOLD`), not by dropping pixels, so thin structures survive and edges are not stair-stepped.
* Where two groups claim the same voxel, the group with the *smaller* structures is painted last (on top).
* Labels are stored as `uint8` (87 ids), which allows ~4x more voxels than before.


In [7]:
def build_lut(gi):
    labels = {k: int(v) for k, v in json.load(open(MODEL_INFO[gi]["dataset_json"]))["labels"].items() if int(v) != 0}
    atlas_ids = [int(a) for a in groups[gi]["atlas_ids"]]
    positional = {loc: atlas_ids[loc - 1] for loc in sorted(labels.values())}
    parsed = {}
    for nm, loc in labels.items():
        hit = [n for n in map(int, re.findall(r"\d+", nm)) if n in atlas_ids]
        if len(hit) == 1: parsed[loc] = hit[0]
    use_parsed = len(parsed) == len(labels) and len(set(parsed.values())) == len(parsed)
    mapping = parsed if use_parsed else positional
    if use_parsed and parsed != positional:
        print(f"  !! group {gi}: class names suggest a DIFFERENT mapping than atlas_ids order. Using names. Verify!")
    lut = np.zeros(256, LABEL_DTYPE)
    for loc, aid in mapping.items(): lut[loc] = aid
    print(f"group {gi}: {'names' if use_parsed else 'positional'} mapping local->atlas: {dict(list(mapping.items())[:6])}{' ...' if len(mapping) > 6 else ''}")
    return lut, mapping

LABEL_DTYPE = np.uint8 if max(ALL_ATLAS_IDS) < 256 else np.uint16
LUTS = {gi: build_lut(gi) for gi in sorted(groups)}
# paint order: groups with the biggest typical structures first, smallest last (= on top). From the training manifest.
PAINT_ORDER = sorted(groups, key=lambda gi: groups[gi]["total_positive_pixels"] / groups[gi]["n_classes"], reverse=True)
print("paint order (big -> small):", PAINT_ORDER)

def pooled_group_slab(gi, zs):
    lut = LUTS[gi][0]
    acc = {}
    for z in zs:
        a = lut[np.array(Image.open(PRED / f"group{gi:02d}" / f"{TAG}_{z:05d}.png"))]
        for l in np.unique(a):
            if l == 0: continue
            m = (a == l).astype(np.float32)
            cov = m if DXY == 1 else cv2.resize(m, (WS, HS), interpolation=cv2.INTER_AREA)
            acc[int(l)] = acc.get(int(l), 0) + cov
    best = np.zeros((HS, WS), np.float32); lab = np.zeros((HS, WS), LABEL_DTYPE)
    for l, c in acc.items():
        c = c / len(zs)
        sel = (c >= FG_THRESHOLD) & (c > best)
        lab[sel] = l; best[sel] = c[sel]
    return lab

label_vol = np.zeros((len(SLABS), HS, WS), LABEL_DTYPE)
t0 = time.time()
for k, zs in enumerate(SLABS):
    for gi in PAINT_ORDER:
        m = pooled_group_slab(gi, zs); sel = m > 0
        label_vol[k][sel] = m[sel]
    if k % 100 == 0: print(f"  slab {k}/{len(SLABS)}  ({time.time()-t0:.0f}s)")
ids_present = [int(i) for i in np.unique(label_vol) if i != 0]
print("label volume", label_vol.shape, label_vol.dtype, "|", len(ids_present), "structure IDs present")
np.save(BASE / f"{TAG}_labels_ds.npy", label_vol)


group 0: names mapping local->atlas: {1: 1, 2: 10, 3: 12, 4: 14, 5: 20, 6: 21} ...
group 1: names mapping local->atlas: {1: 3, 2: 4, 3: 5, 4: 6, 5: 9, 6: 11} ...
group 2: names mapping local->atlas: {1: 15, 2: 16, 3: 17, 4: 22, 5: 24, 6: 25} ...
group 3: names mapping local->atlas: {1: 2, 2: 7, 3: 8, 4: 13, 5: 40, 6: 44} ...
paint order (big -> small): [0, 1, 2, 3]
  slab 0/1059  (0s)
  slab 100/1059  (20s)
  slab 200/1059  (43s)
  slab 300/1059  (69s)
  slab 400/1059  (97s)
  slab 500/1059  (126s)
  slab 600/1059  (157s)
  slab 700/1059  (185s)
  slab 800/1059  (211s)
  slab 900/1059  (236s)
  slab 1000/1059  (258s)
label volume (1059, 911, 563) uint8 | 86 structure IDs present


In [8]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
counts = np.bincount(label_vol.ravel(), minlength=max(ALL_ATLAS_IDS) + 1)
missing = [i for i in ALL_ATLAS_IDS if counts[i] == 0]
print(f"{len(ids_present)} / {len(ALL_ATLAS_IDS)} expected structures were predicted.")
if missing: print("NOT predicted at all (their meshes will be absent):", missing)
unexpected = [i for i in ids_present if i not in set(ALL_ATLAS_IDS)]
assert not unexpected, f"[BUG] label ids that belong to no group: {unexpected}"
print("smallest predicted structures (voxels):", sorted((int(counts[i]), i) for i in ids_present)[:8])

rng = np.random.default_rng(0); pal = rng.uniform(.2, 1, (max(ALL_ATLAS_IDS) + 1, 3)); pal[0] = 0
ks = np.linspace(0, len(SLABS) - 1, 6, dtype=int)
fig, ax = plt.subplots(1, 6, figsize=(24, 5))
for a, k in zip(ax, ks):
    a.imshow(pal[label_vol[k]]); a.set_title(f"slab {k}"); a.axis("off")
plt.tight_layout(); (BASE / "reports").mkdir(exist_ok=True); plt.savefig(BASE / "reports" / "label_preview.png", dpi=80); plt.close()
print("preview:", BASE / "reports" / "label_preview.png")


86 / 87 expected structures were predicted.
NOT predicted at all (their meshes will be absent): [7]
smallest predicted structures (voxels): [(2117, 83), (2188, 66), (3215, 86), (3848, 54), (4226, 78), (4338, 75), (5043, 55), (5189, 81)]
preview: /kaggle/working/cs17_recon_CS17_6521/reports/label_preview.png


## 5. Mesh every structure
per structure: crop → keep real components → fill holes → physical-scale gaussian (extra along z) → marching cubes → Taubin → adaptive decimation (fine + coarse level) → baked ambient occlusion.


In [9]:
from scipy import ndimage as ndi
from skimage.measure import marching_cubes
import trimesh

SP = np.array(SPACING_MM_ZYX, np.float64)

def block_mean(a, s):
    if s == 1: return a.astype(np.float32)
    z, y, x = [(n // s) * s for n in a.shape]
    a = a[:z, :y, :x].reshape(z // s, s, y // s, s, x // s, s)
    return (a.sum(axis=(1, 3, 5), dtype=np.uint32) / float(s ** 3)).astype(np.float32)

def decimate(mesh, target):
    if len(mesh.faces) <= target: return mesh
    try: return mesh.simplify_quadric_decimation(face_count=target)
    except Exception as e:
        print("   decimation failed, keeping full mesh:", e); return mesh

def vertex_ao(verts_mm, normals_mm, field, origin_vox, s, sp):
    """Cheap per-vertex cavity shading from the structure's own blurred occupancy (255 = fully lit)."""
    idx = (verts_mm / SP - origin_vox) / s + (normals_mm * AO_OFFSET_MM) / sp
    occ = ndi.map_coordinates(field, idx.T, order=1, mode="nearest")
    ao = np.clip(1.0 - AO_GAIN * np.maximum(occ - AO_BIAS, 0), 0, 1)
    return np.rint(ao * 255).astype(np.uint8)

def mesh_structure(sid, sl):
    pad = 3
    lo = [max(s.start - pad, 0) for s in sl]; hi = [min(s.stop + pad, n) for s, n in zip(sl, label_vol.shape)]
    crop = label_vol[tuple(slice(a, b) for a, b in zip(lo, hi))] == sid
    nvox = int(crop.sum())
    if nvox < MIN_VOXELS: return None
    s = 1 if crop.size <= MESH_MAX_CROP else int(math.ceil((crop.size / MESH_MAX_CROP) ** (1 / 3)))
    f = block_mean(crop, s); del crop
    b = f >= 0.5
    if not b.any(): b = f >= f.max() * 0.5
    lab, nc = ndi.label(b)
    if nc > 1:                                              # drop floating specks, keep every real part
        sizes = np.asarray(ndi.sum(b, lab, range(1, nc + 1)))
        b = np.isin(lab, [i + 1 for i, z in enumerate(sizes) if z >= max(4, 0.02 * sizes.max())])
    b = ndi.binary_fill_holes(b)
    sp = SP * s
    sig = np.clip(SMOOTH_MM / sp, 0.45, 2.2) * np.array([Z_EXTRA, 1.0, 1.0])
    padded = np.pad(b.astype(np.float32), 2)
    for scale in (1.0, 0.6, 0.35, 0.0):                     # thin structures need less smoothing or they vanish
        g = ndi.gaussian_filter(padded, sig * scale) if scale > 0 else padded
        if g.max() >= 0.55: break
    verts, faces, _, _ = marching_cubes(g, level=0.5, spacing=tuple(sp))
    origin_vox = np.array(lo, np.float64) - 2 * s + (s - 1) / 2.0
    verts = verts + origin_vox * SP
    mesh = trimesh.Trimesh(verts, faces, process=True)
    if len(mesh.faces) < 8: return None
    trimesh.smoothing.filter_taubin(mesh, lamb=0.5, nu=-0.53, iterations=TAUBIN_ITERS)
    target = int(np.clip(FACES_PER_VOX23 * nvox ** (2 / 3), MIN_FACES, MAX_FACES))
    fine = decimate(mesh, target)
    trimesh.smoothing.filter_taubin(fine, lamb=0.5, nu=-0.53, iterations=2)
    field = ndi.gaussian_filter(padded, np.clip(AO_RADIUS_MM / sp, 0.8, 6.0))
    levels = [fine]
    if len(fine.faces) >= 3 * LOD0_MIN_FACES:
        levels.insert(0, decimate(fine, max(LOD0_MIN_FACES, int(LOD0_FRACTION * len(fine.faces)))))
    out = []
    for m in levels:
        v = np.asarray(m.vertices, np.float64); n = np.asarray(m.vertex_normals, np.float64)
        ao = vertex_ao(v, n, field, origin_vox, s, sp)
        v = np.stack([v[:, 2], -v[:, 1], v[:, 0]], 1)       # (z,y,x)mm -> (x,-y,z): proper rotation, winding preserved
        out.append((v.astype(np.float32), np.asarray(m.faces, np.int64), ao))
    return dict(levels=out, nvox=nvox, vol_mm3=float(nvox * SP.prod()))

objs = ndi.find_objects(label_vol.astype(np.int32) if label_vol.dtype != np.uint8 else label_vol)
MESHES = {}
t0 = time.time()
for sid in ids_present:
    sl = objs[sid - 1] if sid - 1 < len(objs) else None
    if sl is None: continue
    r = mesh_structure(sid, sl)
    if r is None: print(f"[skip] id {sid}: too small/thin"); continue
    MESHES[sid] = r
    print(f"[ok] id {sid:>4}: " + " + ".join(f"{len(l[1]):>6}f" for l in r["levels"]) + f"   ({r['nvox']:,} vox)  {time.time()-t0:.0f}s")
print(f"{len(MESHES)} meshes in {time.time()-t0:.0f}s")


[ok] id    1:   4000f +  40000f   (92,200,813 vox)  14s
[ok] id    2:    150f +   1464f   (5,471 vox)  16s
[ok] id    3:   1714f +  17146f   (200,841 vox)  22s
[ok] id    4:   1776f +  17760f   (211,698 vox)  30s
[ok] id    5:    938f +   9394f   (81,445 vox)  36s
[ok] id    6:    990f +   9912f   (88,280 vox)  40s
[ok] id    8:    302f +   3034f   (14,948 vox)  41s
[ok] id    9:   1522f +  15220f   (192,756 vox)  45s
[ok] id   10:   2162f +  21620f   (313,695 vox)  46s
[ok] id   11:   1134f +  11346f   (108,106 vox)  50s
[ok] id   12:   2614f +  26146f   (378,165 vox)  52s
[ok] id   13:    160f +   1600f   (5,734 vox)  54s
[ok] id   14:   4000f +  40000f   (4,281,151 vox)  60s
[ok] id   15:    342f +   3424f   (17,929 vox)  60s
[ok] id   16:    354f +   3550f   (18,934 vox)  62s
[ok] id   17:    920f +   9208f   (79,047 vox)  63s
[ok] id   18:   1010f +  10104f   (243,472 vox)  68s
[ok] id   19:   1242f +  12438f   (124,086 vox)  71s
[ok] id   20:   3998f +  40000f   (1,355,769 vox)  

## 6. Pack the bundle (`meshes.bin` + `manifest.json` + `names.json`)
Layout of `meshes.bin`: all coarse levels first (the viewer shows the whole embryo as soon as they arrive), then fine levels, biggest first.


In [10]:
import colorsys

def zigzag(d):  # int64 -> uint32
    return ((d << 1) ^ (d >> 63)).astype(np.uint32)

def pack_blob(verts, faces, ao, mn, ext):
    q = np.clip(np.rint((verts - mn) / ext * 65535), 0, 65535).astype(np.int64)
    flat = faces.ravel()
    uniq, first = np.unique(flat, return_index=True)
    order = uniq[np.argsort(first)]                        # vertices renumbered by first use -> small index deltas
    remap = np.full(len(verts), -1, np.int64); remap[order] = np.arange(len(order))
    q, ao = q[order], ao[order]; idx = remap[flat]
    vals = np.concatenate([zigzag(np.diff(q[:, a], prepend=0)) for a in range(3)] + [zigzag(np.diff(idx, prepend=0))])
    planes = vals.astype("<u4").view(np.uint8).reshape(-1, 4).T.ravel()   # byte-plane shuffle: zeros cluster -> zlib loves it
    return zlib.compress(planes.tobytes() + ao.tobytes(), 9), len(order), len(faces)

def color_for(sid):
    h = (sid * 0.61803398875) % 1.0
    r, g, b = colorsys.hsv_to_rgb(h, 0.50 + 0.22 * ((sid * 7) % 3) / 2, 0.82 + 0.13 * ((sid * 5) % 2))
    return [int(r * 255), int(g * 255), int(b * 255)]

allv = np.concatenate([lv[0] for m in MESHES.values() for lv in m["levels"]])
BB_MIN = allv.min(0).astype(np.float64); BB_EXT = np.maximum(allv.max(0) - allv.min(0), 1e-9).astype(np.float64)

blobs = []                       # (section, sort_key, sid, lod_index, compressed, n, f)
for sid, m in sorted(MESHES.items()):
    for li, (v, f, ao) in enumerate(m["levels"]):
        comp, n, nf = pack_blob(v, f, ao, BB_MIN, BB_EXT)
        blobs.append((0 if li == 0 else 1, -nf if li else sid, sid, li, comp, n, nf))
blobs.sort(key=lambda b: (b[0], b[1]))
binblob = bytearray(); lods = {sid: [None] * len(m["levels"]) for sid, m in MESHES.items()}
for sec, key, sid, li, comp, n, nf in blobs:
    lods[sid][li] = dict(off=len(binblob), len=len(comp), nv=n, nf=nf); binblob += comp

names_local = OUT / "names.json"
if not names_local.exists():                                 # template only; an existing names.json is never touched
    json.dump({str(s): "" for s in sorted(MESHES)}, open(names_local, "w"), indent=1)

structs = [dict(id=sid, color=color_for(sid), voxels=m["nvox"], volume_mm3=round(m["vol_mm3"], 5), lods=lods[sid])
           for sid, m in sorted(MESHES.items())]
BUNDLE_MANIFEST = dict(version=2, specimen=SPECIMEN_ID, title=f"Embryo {SPECIMEN_ID}", units="mm",
                       bbox_min=BB_MIN.tolist(), bbox_ext=BB_EXT.tolist(), ao_strength=AO_STRENGTH,
                       files=dict(bin="meshes.bin", names="names.json"), bin_bytes=len(binblob),
                       n_faces_fine=int(sum(m["levels"][-1][1].shape[0] for m in MESHES.values())),
                       models={str(gi): dict(checkpoint=mi["ckpt"], epoch=mi["ckpt_epoch"]) for gi, mi in MODEL_INFO.items()},
                       label_voxel_um=[round(v * 1000, 3) for v in SPACING_MM_ZYX],
                       generated=time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), structures=structs)
(OUT / "meshes.bin").write_bytes(bytes(binblob))
json.dump(BUNDLE_MANIFEST, open(OUT / "manifest.json", "w"))
first_section = sum(s["lods"][0]["len"] for s in structs)
print(f"{len(structs)} structures | {BUNDLE_MANIFEST['n_faces_fine']:,} fine triangles | meshes.bin {len(binblob)/1e6:.2f} MB "
      f"(first paint after {first_section/1e6:.2f} MB)")


86 structures | 1,225,624 fine triangles | meshes.bin 7.58 MB (first paint after 0.74 MB)


In [11]:
if PUSH_TO_HF:
    api.create_repo(repo_id=HF_SHARED_REPO_ID, repo_type="dataset", private=False, exist_ok=True)
    prefix = f"viewers/{SPECIMEN_ID}"
    remote = set(api.list_repo_files(HF_SHARED_REPO_ID, repo_type="dataset"))
    keep_remote_names = f"{prefix}/names.json" in remote
    if keep_remote_names:
        print("names.json already exists in the dataset -> NOT overwritten")
    api.upload_folder(folder_path=str(OUT), repo_id=HF_SHARED_REPO_ID, repo_type="dataset", path_in_repo=prefix,
                      ignore_patterns=["names.json"] if keep_remote_names else None,
                      commit_message=f"{SPECIMEN_ID} viewer bundle")
    base = f"https://huggingface.co/datasets/{HF_SHARED_REPO_ID}/resolve/main/{prefix}/"
    print("\nUploaded. Use this on your page:\n")
    print(f'<script type="module" src="embryo-viewer.js"></script>\n<embryo-viewer src="{base}" style="width:100%;height:600px"></embryo-viewer>')
else:
    print("PUSH_TO_HF=False - bundle is in", OUT)



Uploaded. Use this on your page:

<script type="module" src="embryo-viewer.js"></script>
<embryo-viewer src="https://huggingface.co/datasets/senku111/embryo-3d/resolve/main/viewers/CS17-6521/" style="width:100%;height:600px"></embryo-viewer>


## Notes
* **Names:** edit `viewers/<SPECIMEN>/names.json` directly in the dataset (`{"17": "Heart", ...}` or `{"17": {"name": "Heart", "color": "#c0392b", "group": "Circulatory"}}`). No rebuild is needed.
* **IDs** are the Atlas label ids; they are never renumbered. A structure that is too small/thin simply has no mesh.
* If the result is too heavy for your page: lower `MAX_FACES` / `FACES_PER_VOX23`; too coarse: raise them (or `MAX_VOXELS`).
